# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [ ]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)

In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [4]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [11]:
from pyspark.sql.functions import monotonically_increasing_id
df_with_id = df_trips.withColumn("trip_id", monotonically_increasing_id())
# I used the monotonically_increasing_id() function to generate a generate a unique key for each row

In [ ]:
from pyspark.sql.functions import col
df_with_id.orderBy(col("passenger_count").desc()).show(1)
# I ordered my data by descending order of passenger_count, and I showed only the first row

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|    trip_id|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|       5.0|                 N|          68|          68|           1

In [ ]:
from pyspark.sql.functions import avg
df_with_id.select(avg("passenger_count")).show()
# I used avg() function to calculate the passenger count average

+--------------------+
|avg(passenger_count)|
+--------------------+
|  1.5670317144945614|
+--------------------+



In [23]:
from pyspark.sql.functions import min, max
df_with_id.select(min("trip_distance"), max("trip_distance")).show()
# I used min, max functions to get the min and max distance 

+------------------+------------------+
|min(trip_distance)|max(trip_distance)|
+------------------+------------------+
|               0.0|             831.8|
+------------------+------------------+



In [25]:
from pyspark.sql.functions import col, unix_timestamp, min, max

# Compute the duration in minutes (or in seconds by removing the / 60)
df_with_duration = df_with_id.withColumn(
    "duration_minutes",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
)

df_with_duration.select(min("duration_minutes"), max("duration_minutes")).show()

+---------------------+---------------------+
|min(duration_minutes)|max(duration_minutes)|
+---------------------+---------------------+
|             -84280.5|    43648.01666666667|
+---------------------+---------------------+



In [28]:
# there is outbound values so we have to clean the data
df_clean = df_with_duration.filter(
    (col("duration_minutes") > 0) & 
    (col("duration_minutes") <= 720) &
    (col("trip_distance") > 0)
)
df_clean.select(min("duration_minutes"), max("duration_minutes")).show()

+---------------------+---------------------+
|min(duration_minutes)|max(duration_minutes)|
+---------------------+---------------------+
| 0.016666666666666666|                719.8|
+---------------------+---------------------+



In [29]:
# it sounds good now !

In [32]:
from pyspark.sql.functions import to_date, count, col
# creation of a dataframe aggregated by day
daily_trips = df_clean.withColumn("pickup_date", to_date("tpep_pickup_datetime")) \
    .groupBy("pickup_date") \
    .agg(count("*").alias("trip_count"))
daily_trips.show()
# we used to_date to convert a full date type into a day type
# we used groupBy to group the dataframe by the new colomn pickup_date
# we used agg to count all occurences of a single day (we named it trip_count)

+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-02-23|         1|
| 2009-01-01|        40|
| 2019-01-07|    226697|
| 2019-01-08|    235035|
| 2019-01-28|    238900|
| 2019-03-17|         2|
| 2019-04-28|         3|
| 2019-01-30|    274377|
| 2019-01-26|    269360|
| 2019-06-10|         2|
| 2018-12-30|         7|
| 2019-02-01|        61|
| 2019-05-20|         1|
| 2019-01-20|    201019|
| 2019-03-19|         3|
| 2019-01-22|    252614|
| 2019-01-19|    234091|
| 2019-07-23|         1|
| 2019-01-05|    234224|
| 2019-01-03|    221652|
+-----------+----------+
only showing top 20 rows


In [34]:
daily_trips.orderBy(col("trip_count").desc()).show(1)
daily_trips.orderBy(col("trip_count").asc()).show(1)

+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-01-25|    289786|
+-----------+----------+
only showing top 1 row
+-----------+----------+
|pickup_date|trip_count|
+-----------+----------+
| 2019-05-20|         1|
+-----------+----------+
only showing top 1 row


In [35]:
# the busiest day is the 25/01/2019 and the slowest is the 20/05/2019

In [36]:
from pyspark.sql.functions import hour, count, col

# Extract pickup hour (0 to 23) and aggregate trip counts
hourly_trips = (
    df_clean
    .withColumn("pickup_hour", hour("tpep_pickup_datetime"))
    .groupBy("pickup_hour")
    .agg(count("*").alias("trip_count"))
)

# Display the busiest hour of the day
print("Busiest hour:")
hourly_trips.orderBy(col("trip_count").desc()).show(1)

# Display the slowest hour of the day
print("Slowest hour:")
hourly_trips.orderBy(col("trip_count").asc()).show(1)

# View full distribution sorted chronologically
hourly_trips.orderBy("pickup_hour").show(24)

Busiest hour:
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|         18|    511222|
+-----------+----------+
only showing top 1 row
Slowest hour:
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          4|     60037|
+-----------+----------+
only showing top 1 row
+-----------+----------+
|pickup_hour|trip_count|
+-----------+----------+
|          0|    205293|
|          1|    147304|
|          2|    107788|
|          3|     76657|
|          4|     60037|
|          5|     74058|
|          6|    176054|
|          7|    301420|
|          8|    370460|
|          9|    362758|
|         10|    358096|
|         11|    371961|
|         12|    397465|
|         13|    400257|
|         14|    428840|
|         15|    448050|
|         16|    416248|
|         17|    464140|
|         18|    511222|
|         19|    471619|
|         20|    419869|
|         21|    406522|
|         22|    365938|
|         23|    279187|
+

In [38]:
from pyspark.sql.functions import to_date, date_format, dayofweek, count, avg, col

# Step 1: Count trips per individual calendar date
daily_counts = (
    df_clean
    .withColumn("trip_date", to_date("tpep_pickup_datetime"))
    .groupBy("trip_date")
    .agg(count("*").alias("daily_total"))
)

# Step 2: Extract day name and day number using native dayofweek()
avg_by_day = (
    daily_counts
    .withColumn("day_of_week", date_format("trip_date", "EEEE"))  # e.g., 'Monday'
    .withColumn("day_num", dayofweek("trip_date"))                 # 1 = Sun, 2 = Mon, ..., 7 = Sat
    .groupBy("day_of_week", "day_num")
    .agg(avg("daily_total").alias("avg_daily_trips"))
)

# Busiest day on average
print("Busiest day of the week on average:")
avg_by_day.orderBy(col("avg_daily_trips").desc()).select("day_of_week", "avg_daily_trips").show(1)

# Slowest day on average
print("Slowest day of the week on average:")
avg_by_day.orderBy(col("avg_daily_trips").asc()).select("day_of_week", "avg_daily_trips").show(1)

# Full breakdown
avg_by_day.orderBy("day_num").select("day_of_week", "avg_daily_trips").show()

Busiest day of the week on average:
+-----------+------------------+
|day_of_week|   avg_daily_trips|
+-----------+------------------+
|   Thursday|224034.83333333334|
+-----------+------------------+
only showing top 1 row
Slowest day of the week on average:
+-----------+---------------+
|day_of_week|avg_daily_trips|
+-----------+---------------+
|     Monday|        89915.9|
+-----------+---------------+
only showing top 1 row
+-----------+------------------+
|day_of_week|   avg_daily_trips|
+-----------+------------------+
|     Sunday|         106354.75|
|     Monday|           89915.9|
|    Tuesday|          119691.0|
|  Wednesday|179043.57142857142|
|   Thursday|224034.83333333334|
|     Friday|          215330.8|
|   Saturday|142881.14285714287|
+-----------+------------------+



In [39]:
from pyspark.sql.functions import corr, col

# Filter for credit card trips with valid positive amounts
df_tips = df_clean.filter(
    (col("payment_type") == 1) & 
    (col("tip_amount") >= 0) & 
    (col("passenger_count") > 0)
)

# Compute correlations with tip_amount
corr_distance = df_tips.stat.corr("tip_amount", "trip_distance")
corr_passengers = df_tips.stat.corr("tip_amount", "passenger_count")

print(f"Correlation between Tip Amount and Trip Distance: {corr_distance:.4f}")
print(f"Correlation between Tip Amount and Passenger Count: {corr_passengers:.4f}")

Correlation between Tip Amount and Trip Distance: 0.7022
Correlation between Tip Amount and Passenger Count: 0.0107


In [40]:
from pyspark.sql.functions import col

# Select key identifying columns and order descending
(
    df_clean
    .select("trip_id", "extra", "fare_amount", "total_amount", "tpep_pickup_datetime")
    .orderBy(col("extra").desc())
    .show(5, truncate=False)
)

+-----------+-----+-----------+------------+--------------------+
|trip_id    |extra|fare_amount|total_amount|tpep_pickup_datetime|
+-----------+-----+-----------+------------+--------------------+
|68719787788|18.5 |52.0       |88.3        |2019-01-02 16:33:28 |
|68721931822|18.5 |49.0       |96.36       |2019-01-11 16:08:48 |
|68719611285|18.5 |39.5       |70.8        |2019-01-01 16:09:32 |
|68720019939|18.5 |61.0       |92.3        |2019-01-03 18:32:36 |
|68720025044|18.5 |47.5       |94.56       |2019-01-03 18:19:33 |
+-----------+-----+-----------+------------+--------------------+
only showing top 5 rows


### Data Anomalies and Outliers

* **Negative Durations (e.g., -84,280 min):** Dropoff timestamp occurs before pickup timestamp, caused by clock desynchronization or meter software bugs.
* **Extreme Durations (> 40,000 min / ~30 days):** Drivers forgot to turn off the taximeter, leaving the session open until month-end.
* **Negative Fares & Extras (`total_amount < 0`):** Represent refunds, cancellations, or transaction disputes rather than real rides.
* **Zero Distance with Positive Fare (`trip_distance = 0`):** Trips cancelled on arrival, passenger wait times, or flat-rate charges with GPS inactive.
* **Extreme `extra` Charges:** Standard NYC TLC extra surcharges are strictly $0.50 (night) or $1.00 (rush hour). High values indicate manual overrides or data-entry errors.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing